# EventFlow — Minimal Reproduction

A self-contained **PyTorch** reproduction of the balanced-coupling flow-matching method for temporal point processes (TPPs).

We learn a velocity field that transports a **uniform reference** TPP onto a **clustered data** TPP, generate samples by integrating the ODE, and check that (a) the event-count distribution is preserved and (b) generated event times are much closer to the data clusters than the uniform reference.

This notebook runs at **lightweight scale** (fewer samples / epochs) so it finishes in seconds on a CPU. It reproduces the *qualitative* behavior; the full configuration (10k train samples, 200 epochs) yields the exact numbers reported in the post.

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = "cpu"

## Configuration

Reduced for speed. The full run that produced the post's numbers used `TRAIN_SIZE=10000`, `EPOCHS=200`, `ODE_STEPS=100`, `N_GEN=2000`.

In [ ]:
T = 10.0                       # observation window [0, T]
SIGMA = 0.05                   # interpolant noise std
SIGMA_DATA = 0.35              # data cluster width
COUNT_PROBS = {1: 0.2, 2: 0.5, 3: 0.3}
CENTERS = {1: [5.0], 2: [3.0, 7.0], 3: [2.0, 5.0, 8.0]}
N_MAX = max(COUNT_PROBS)

TRAIN_SIZE, VAL_SIZE = 2000, 500   # full run: 10000 / 1000
BATCH_SIZE = 128
EPOCHS = 60                        # full run: 200
LR = 1e-3
ODE_STEPS = 50                     # full run: 100
N_GEN = 2000

## Synthetic data: the clustered TPP $\mu_1$

Sample a count $n\sim\mu_1(n)$, then each event $t^k\sim\mathcal N(c_k,\sigma_{\text{data}}^2)$, clipped to $[0,T]$ and sorted.

In [ ]:
counts = np.array(list(COUNT_PROBS.keys()))
probs = np.array(list(COUNT_PROBS.values()))

def sample_data_sequence():
    n = int(np.random.choice(counts, p=probs))
    c = np.array(CENTERS[n])
    t = np.clip(c + SIGMA_DATA * np.random.randn(n), 0.0, T)
    return np.sort(t).astype(np.float32)

train_data = [sample_data_sequence() for _ in range(TRAIN_SIZE)]
val_data = [sample_data_sequence() for _ in range(VAL_SIZE)]
print("example sequences:", [s.round(2).tolist() for s in train_data[:4]])

## Balanced coupling + noisy linear interpolant

For each data sequence $\gamma_1$, draw a uniform reference $\gamma_0$ with the **same count**, pick $s\sim U(0,1)$, and form
$$\hat\gamma_s=(1-s)\gamma_0+s\gamma_1+\sigma\,\varepsilon,\qquad \text{target } v=\gamma_1-\gamma_0.$$
Sequences are padded to `N_MAX` with a mask.

In [ ]:
def sample_reference_like(gamma_1):
    n = gamma_1.shape[0]
    return np.sort(np.random.rand(n) * T).astype(np.float32)

def collate_balanced_batch(seqs):
    B = len(seqs)
    gamma_hat = np.zeros((B, N_MAX), np.float32)
    target_v = np.zeros((B, N_MAX), np.float32)
    mask = np.zeros((B, N_MAX), np.float32)
    s_all = np.zeros((B, 1), np.float32)
    for i, g1 in enumerate(seqs):
        n = g1.shape[0]
        g0 = sample_reference_like(g1)
        s = np.random.rand()
        gs = (1.0 - s) * g0 + s * g1
        ghat = np.clip(gs + SIGMA * np.random.randn(n), 0.0, T)
        gamma_hat[i, :n] = ghat
        target_v[i, :n] = g1 - g0
        mask[i, :n] = 1.0
        s_all[i, 0] = s
    to = lambda a: torch.from_numpy(a).to(device)
    return to(gamma_hat), to(s_all), to(target_v), to(mask)

## Vector field: pointwise MLP with context

Per-event features $[t_k,\,s,\,t_k/T,\,N/N_{\max},\,\bar t/T]$ concatenated with a sinusoidal embedding of $s$; one scalar velocity per event.

In [ ]:
def sinusoidal_embedding(s, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000.0) * torch.arange(half, device=s.device) / max(half - 1, 1))
    a = s * freqs  # [B, half]
    return torch.cat([torch.sin(a), torch.cos(a)], dim=-1)

class VectorFieldMLP(nn.Module):
    def __init__(self, hidden=128, layers=3, time_emb=32):
        super().__init__()
        self.time_emb = time_emb
        in_dim = 5 + time_emb
        net = [nn.Linear(in_dim, hidden), nn.SiLU()]
        for _ in range(layers - 1):
            net += [nn.Linear(hidden, hidden), nn.SiLU()]
        net += [nn.Linear(hidden, 1)]
        self.net = nn.Sequential(*net)

    def forward(self, gamma, s, mask):
        B, Nm = gamma.shape
        count = mask.sum(-1, keepdim=True)                       # [B,1]
        masked_mean = (gamma * mask).sum(-1, keepdim=True) / count.clamp_min(1.0)
        s_exp = s.expand(B, Nm)
        feats = torch.stack([
            gamma, s_exp, gamma / T,
            (count / N_MAX).expand(B, Nm),
            (masked_mean / T).expand(B, Nm),
        ], dim=-1)                                               # [B,Nm,5]
        emb = sinusoidal_embedding(s, self.time_emb)             # [B,emb]
        emb = emb.unsqueeze(1).expand(B, Nm, self.time_emb)
        x = torch.cat([feats, emb], dim=-1)
        v = self.net(x).squeeze(-1)                              # [B,Nm]
        return v * mask

def masked_mse(pred, target, mask):
    se = ((pred - target) ** 2) * mask
    return se.sum() / mask.sum().clamp_min(1.0)

model = VectorFieldMLP().to(device)
print("parameters:", sum(p.numel() for p in model.parameters()))

## Training (masked MSE)

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=LR)

def epoch_pass(data, train):
    model.train(train)
    order = np.random.permutation(len(data)) if train else np.arange(len(data))
    total, nb = 0.0, 0
    for i in range(0, len(data), BATCH_SIZE):
        seqs = [data[j] for j in order[i:i + BATCH_SIZE]]
        gamma, s, tv, mask = collate_balanced_batch(seqs)
        pred = model(gamma, s, mask)
        loss = masked_mse(pred, tv, mask)
        if train:
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        total += loss.item(); nb += 1
    return total / nb

for ep in range(1, EPOCHS + 1):
    tr = epoch_pass(train_data, True)
    with torch.no_grad():
        va = epoch_pass(val_data, False)
    if ep == 1 or ep % 10 == 0 or ep == EPOCHS:
        print(f"epoch {ep:3d}  train_mse={tr:.4f}  val_mse={va:.4f}")

## ODE sampling (explicit Euler)

Fix the counts by sampling from $\mu_1(n)$, start from a uniform reference, integrate $d\gamma/ds=v_\theta$, then clip and sort.

In [ ]:
def build_reference(n_list):
    B = len(n_list)
    gamma = np.zeros((B, N_MAX), np.float32)
    mask = np.zeros((B, N_MAX), np.float32)
    for i, n in enumerate(n_list):
        gamma[i, :n] = np.sort(np.random.rand(n) * T)
        mask[i, :n] = 1.0
    return torch.from_numpy(gamma).to(device), torch.from_numpy(mask).to(device)

@torch.no_grad()
def sample_with_ode(n_list, num_steps=ODE_STEPS, return_traj=False):
    gamma, mask = build_reference(n_list)
    B = gamma.shape[0]
    dt = 1.0 / num_steps
    traj = [gamma.clone()]
    for j in range(num_steps):
        s = torch.full((B, 1), j / num_steps, device=device)
        gamma = (gamma + dt * model(gamma, s, mask)).clamp(0.0, T) * mask
        if return_traj:
            traj.append(gamma.clone())
    out = gamma.clone()
    for i in range(B):
        n = int(mask[i].sum())
        if n > 0:
            out[i, :n] = torch.sort(gamma[i, :n]).values
    return (out, mask, torch.stack(traj)) if return_traj else (out, mask)

gen_counts = list(np.random.choice(counts, size=N_GEN, p=probs))
gen, gen_mask = sample_with_ode(gen_counts)

## Metrics: count L1, mean-time error, 1-D Wasserstein

Compared against the uniform-reference baseline. The Wasserstein-1 distance between two equal-size samples is the mean absolute difference of their sorted values.

In [ ]:
def w1(a, b):
    a, b = np.sort(np.asarray(a)), np.sort(np.asarray(b))
    m = min(len(a), len(b))
    if m == 0:
        return 0.0
    qa = np.interp(np.linspace(0, 1, 200), np.linspace(0, 1, len(a)), a)
    qb = np.interp(np.linspace(0, 1, 200), np.linspace(0, 1, len(b)), b)
    return float(np.mean(np.abs(qa - qb)))

def by_nk(seqs_or_tensor, masks=None):
    """Collect event times grouped by (count, index)."""
    d = {}
    if masks is None:  # list of numpy sequences
        for g in seqs_or_tensor:
            n = len(g)
            for k in range(n):
                d.setdefault((n, k), []).append(float(g[k]))
    else:              # padded tensor + mask
        g = seqs_or_tensor.cpu().numpy(); m = masks.cpu().numpy()
        for i in range(g.shape[0]):
            n = int(m[i].sum())
            for k in range(n):
                d.setdefault((n, k), []).append(float(g[i, k]))
    return d

# count-distribution L1
def count_hist(n_list):
    h = {n: 0 for n in counts}
    for n in n_list:
        h[int(n)] += 1
    tot = sum(h.values())
    return {n: h[n] / tot for n in counts}

data_counts = [len(g) for g in val_data]
gh, dh = count_hist([int(m.sum()) for m in gen_mask.cpu().numpy()]), count_hist(data_counts)
count_l1 = sum(abs(gh[n] - dh[n]) for n in counts)

data_nk = by_nk(val_data)
gen_nk = by_nk(gen, gen_mask)
ref_counts = list(np.random.choice(counts, size=N_GEN, p=probs))
ref_seqs = [np.sort(np.random.rand(int(n)) * T) for n in ref_counts]
ref_nk = by_nk(ref_seqs)

keys = sorted(set(data_nk) & set(gen_nk) & set(ref_nk))
mt_gen = np.mean([abs(np.mean(gen_nk[k]) - np.mean(data_nk[k])) for k in keys])
mt_ref = np.mean([abs(np.mean(ref_nk[k]) - np.mean(data_nk[k])) for k in keys])
w_gen = np.mean([w1(gen_nk[k], data_nk[k]) for k in keys])
w_ref = np.mean([w1(ref_nk[k], data_nk[k]) for k in keys])

print(f"count-distribution L1 error : {count_l1:.3f}")
print(f"mean-time error  generated  : {mt_gen:.3f}   reference: {mt_ref:.3f}")
print(f"wasserstein      generated  : {w_gen:.3f}   reference: {w_ref:.3f}")

## Plots: counts, pooled event times, flow trajectories

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

# (1) count distribution
x = np.array(counts)
ax[0].bar(x - 0.2, [dh[n] for n in counts], 0.4, label="data")
ax[0].bar(x + 0.2, [gh[n] for n in counts], 0.4, label="generated")
ax[0].set_title("event-count distribution"); ax[0].set_xticks(x); ax[0].legend()

# (2) pooled event times
pool = lambda seqs: np.concatenate([np.asarray(s).ravel() for s in seqs])
ref_all = pool(ref_seqs)
data_all = pool(val_data)
gen_all = np.concatenate([g[:int(m.sum())] for g, m in zip(gen.cpu().numpy(), gen_mask.cpu().numpy())])
bins = np.linspace(0, T, 41)
ax[1].hist(ref_all, bins, alpha=0.5, density=True, label="reference")
ax[1].hist(data_all, bins, alpha=0.5, density=True, label="data")
ax[1].hist(gen_all, bins, alpha=0.5, density=True, label="generated")
ax[1].set_title("pooled event times"); ax[1].legend()

# (3) flow trajectories for a few 3-event sequences
_, m3, traj = sample_with_ode([3] * 6, return_traj=True)
ss = np.linspace(0, 1, traj.shape[0])
tr = traj.cpu().numpy()
for i in range(tr.shape[1]):
    for k in range(3):
        ax[2].plot(ss, tr[:, i, k], color="C0", alpha=0.6)
for c in CENTERS[3]:
    ax[2].axhline(c, color="C1", ls="--", alpha=0.5)
ax[2].set_title("flow trajectories (n=3)"); ax[2].set_xlabel("flow time s"); ax[2].set_ylabel("event time t")

plt.tight_layout(); plt.show()

## What to expect

Even at this reduced scale you should see:

- **Counts preserved** — the count-distribution L1 error is tiny, because counts are sampled before the ODE and the flow never adds or removes events.
- **Reference → data transport** — generated mean-time error and Wasserstein distance are well below the uniform-reference baseline, and the pooled-time histogram develops peaks at the cluster centers $[5]$, $[3,7]$, $[2,5,8]$.
- **Clean trajectories** — events fan from their uniform start positions toward the dashed cluster centers as $s:0\to1$.

Scaling up to `TRAIN_SIZE=10000`, `EPOCHS=200`, `ODE_STEPS=100`, `N_GEN=2000` reproduces the post's numbers: count L1 $\approx0.048$, mean-time error $\approx0.159$ (vs $0.236$ reference), Wasserstein $\approx0.217$ (vs $1.595$ reference).